# 07. 면접 피드백 생성 (영상·텍스트 입력) — v5 설정

06(v4)과 같은 흐름에서 **검색 임베딩만 MiniLM → Gemini 임베딩 API(gemini-embedding-001)**로 바꾼 버전입니다. 06은 그대로 둡니다.

| | 06 (v4) | 이 노트북 (v5) |
|---|---|---|
| 참고자료 DB | 강의 영상 74개, 567청크 (`rag_v2.py`, `interview_rag_v2`) | 같은 청크 (`rag_v3.py`, `interview_rag_v3`) |
| 임베딩 | MiniLM — 128토큰까지만 읽어 청크 89%가 잘림 | **gemini-embedding-001** — 2,048토큰, 문서/질문 구분 |
| 검색 질의 · 검색 방식 · 예시 · 프롬프트 | 질의 정제 · 키워드+임베딩 RRF · 비슷한 답변 5개 · slide19 | 같음 |

31쌍 판정(Claude): 검색 근거 관련도 v4 47% → v5 55%, 코치 지적 재현율 51% → 52% (피드백 정확도는 같고 근거가 더 정확해짐)
주제 질문 14개 정답 강의 Hit@3: 10 → 14 (`embed_compare.py`)

In [ ]:
%pip install google-genai chromadb sentence-transformers scikit-learn python-dotenv --quiet

In [ ]:
import os, re, time, random
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

BASE = Path(r'C:\Users\82105\OneDrive\바탕 화면\interview-coach')
PROJ = BASE.parent / '프로젝트3(면접)'
FB_DIR = BASE / 'data' / 'feedback_data'   # Few-Shot용

load_dotenv(PROJ / '.env')
client = genai.Client(api_key=os.environ.get('GOOGLE_API_KEY', ''))

# 발표자료 기준 모델은 gemini-3.1-pro-preview (무료 등급은 한도 0 → 결제 필요)
MODEL = 'gemini-3.5-flash'
# 생각(thinking) 토큰도 출력 한도에 포함되므로 넉넉히 둔다 (1,200이면 답이 잘리거나 비어 나올 수 있음)
MAX_TOKENS = 4096
print(f'설정 완료 | 모델: {MODEL}')

## 피드백 예시 (Few-Shot) — 모의면접 답변 ↔ 코치 피드백 31쌍

In [ ]:
pairs = []
for grp, ivd, fbd in (('1~5', 'Interview_text', 'Feedback_text'), ('11~16', 'interview_text', 'feedback_text')):
    for iv in sorted((FB_DIR / grp / ivd).glob('*.txt')):
        fb = FB_DIR / grp / fbd / iv.name
        if fb.exists():
            pairs.append({'interview': iv.read_text(encoding='utf-8').strip(),
                          'feedback': fb.read_text(encoding='utf-8').strip()})
text_dir = FB_DIR / '6~10' / 'text'
AF_MARK = re.compile(r'^\ufeff?\s*([AF])\.?\s*$')          # 6~10 인터뷰 파일은 A.(답변)·F.(코치 피드백) 블록이 번갈아 있다
ANSWER_LABEL = re.compile(r'^\ufeff?\s*답변\s*\d+\.')        # 평가가 담긴 소제목 줄
def answers_only(text):
    out, keep = [], True
    for line in text.splitlines():
        m = AF_MARK.match(line)
        if m:
            keep = m.group(1) == 'A'
            continue
        if keep and not ANSWER_LABEL.match(line):
            out.append(line)
    return '\n'.join(out).strip()
for iv in sorted(text_dir.glob('*인터뷰*.txt')):
    n = re.match(r'(\d+)', iv.name)
    if not n:
        continue
    fb = '\n\n'.join(f.read_text(encoding='utf-8').strip() for f in sorted(text_dir.glob(f'{n.group(1)}_[0-9]*.txt')))
    pairs.append({'interview': answers_only(iv.read_text(encoding='utf-8')), 'feedback': fb})
pairs = [p for p in pairs if p['interview'] and p['feedback']]
print(f'Few-Shot 예시: {len(pairs)}쌍')

## 참고자료 검색 · 예시 고르기

In [ ]:
import json
from rag_v3 import build_db, HybridRetrieverV3
from prompts_v2 import SYSTEM_PROMPT_V1, VIDEO_RULE
from pipeline_v3 import ExampleSelector, rewrite_query

build_db(client)                    # 최초 1회 구축(Gemini 임베딩 API, 약 $0.02), 이후엔 있는 DB 사용
rag = HybridRetrieverV3(client)     # 질문도 같은 Gemini 임베딩으로 검색
selector = ExampleSelector(pairs)   # 입력 답변과 비슷한 답변-피드백 쌍 고르기

In [ ]:
def build_contents(shots):
    contents = []
    for s in shots:   # Q./A. 형식 (발표자료 slide12 방식)
        contents.append(types.Content(role='user', parts=[types.Part.from_text(text=f'Q.\n{s["interview"]}')]))
        contents.append(types.Content(role='model', parts=[types.Part.from_text(text=f'A.\n{s["feedback"]}')]))
    return contents


def rag_block(hits):
    return '--- 참고 자료 (면접왕 이형 강의) ---\n' + '\n\n'.join(f'[참고{i+1}] {h["text"]}' for i, h in enumerate(hits))


def generate(contents, video=False):
    system = SYSTEM_PROMPT_V1 + ('\n\n영상 입력 규칙:' + VIDEO_RULE if video else '')
    resp = client.models.generate_content(model=MODEL, contents=contents, config=types.GenerateContentConfig(
        system_instruction=system, temperature=0.4, max_output_tokens=MAX_TOKENS))
    # 예시를 Q./A. 형식으로 넣어서 답변 앞에 'A.'를 붙이는 경우가 있어 떼어 낸다
    return re.sub(r'^\s*A\.\s*', '', resp.text or '')


def get_feedback(answer, n_shots=5, n_rag=4):
    '''텍스트 입력: 답변 → 검색용 질문 정제 → 하이브리드 검색 + 비슷한 답변 예시 → 피드백'''
    query = rewrite_query(client, MODEL, answer)
    hits = rag.search(query, n_rag)
    contents = build_contents(selector.select(answer, n_shots))
    contents.append(types.Content(role='user', parts=[types.Part.from_text(text=f'Q.\n{answer}\n\n{rag_block(hits)}')]))
    return {'feedback': generate(contents), 'query': query, 'rag_sources': [h['source'] for h in hits]}


def get_video_feedback(video_path, n_shots=5, n_rag=4):
    '''영상 입력: 업로드 → 답변 요약·검색용 질문 → 하이브리드 검색 + 비슷한 답변 예시 → 영상 + 예시 + 근거 + 규칙 → [MM:SS] 피드백'''
    video = client.files.upload(file=str(video_path), config={'mime_type': 'video/mp4'})
    while video.state.name == 'PROCESSING':
        time.sleep(5)
        video = client.files.get(name=video.name)
    if video.state.name == 'FAILED':
        raise RuntimeError(f'영상 처리 실패: {video.state}')

    pre = client.models.generate_content(model=MODEL, contents=[video,
        '이 면접 영상에서 지원자가 한 답변을 3~4문장으로 요약하고, 이 답변에 피드백하려면 면접 코칭 강의에서 찾아볼 주제를 '
        '검색용 질문 한 문장으로 써줘. JSON으로만 답해: {"summary": "...", "query": "..."}'],
        config=types.GenerateContentConfig(max_output_tokens=MAX_TOKENS))
    m = re.search(r'\{.*\}', pre.text or '', re.S)
    info = json.loads(m.group(0)) if m else {'summary': pre.text or '', 'query': pre.text or ''}
    hits = rag.search(info.get('query', ''), n_rag)
    contents = build_contents(selector.select(info.get('summary', ''), n_shots))
    contents.append(types.Content(role='user', parts=[
        types.Part.from_uri(file_uri=video.uri, mime_type='video/mp4'),
        types.Part.from_text(text=f'Q.\n위 면접 영상을 분석해줘.\n\n{rag_block(hits)}')]))
    feedback = generate(contents, video=True)
    client.files.delete(name=video.name)   # 면접 영상은 개인 정보라 사용 후 서버에서 지운다
    return {'feedback': feedback, 'summary': info.get('summary', ''), 'query': info.get('query', ''),
            'rag_sources': [h['source'] for h in hits]}

print('준비 완료')

## 텍스트 입력

In [ ]:
ANSWER = '''안녕하세요. 저는 성실하고 책임감 있는 사람입니다.
학교에서 열심히 공부했고, 팀 프로젝트도 많이 해봤습니다.
이 회사에 오고 싶어서 지원했습니다. 잘 부탁드립니다.'''

result = get_feedback(ANSWER)
print('검색용 질문:', result['query']); print('참고 강의:', result['rag_sources'])
print(result['feedback'])

## 영상 입력

In [ ]:
VIDEO_PATH = r'C:\Users\82105\OneDrive\바탕 화면\drive_unzip\최종테스트_영상\21.mp4'

result = get_video_feedback(VIDEO_PATH)
print('답변 요약:', result['summary'])
print('검색용 질문:', result['query']); print('참고 강의:', result['rag_sources'])
print(result['feedback'])